# The closing line: did the market ever come round to the models?

Companion notebook to the [closing-line report](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/clv/README.md) of [Jan's Matchday](https://jhvl88.github.io/matchday/).
football-data.co.uk records two prices for every match: one a few days before, one at kick-off. This notebook asks whether prices drifted toward the models in between, and what the site's paper-betting rule would have earned at the early price.
The questions and rules were [fixed beforehand](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/clv/PREREGISTRATION.md).

1. **Source data** for five leagues and one match table per league.
2. **Forecasts** as published: the three models on all five leagues, and the Bundesliga's xG models.
3. **Grades**: the slope of line move on edge, the rule's closing-line value and realised return.

**Runtime:** about two minutes. Nothing is refitted, so no GPU is needed; [`leagues.ipynb`](https://github.com/JHvL88/matchday/blob/main/analysis/backtest/leagues/leagues.ipynb) refits the models themselves.

Set `LATEST = True` to grade today's source files and forecasts; the numbers then drift from the report as matches are added.

In [ ]:
LATEST = False         # True: the newest source files and forecasts, not the ones the report graded

import json, os, subprocess, sys
REPO = "https://github.com/JHvL88/matchday"
BRANCH = os.environ.get("MATCHDAY_BRANCH", "main")
LEAGUES = ["GER", "ENG", "ESP", "ITA", "FRA"]
SOURCE_COMMIT = "08f400c56187a033afd7bc4d50a8e78a780bd154"     # source-data commit behind the report
RESULTS_COMMIT = "f18b825b8a79ffc03435d82f0f3af4e655cad9eb"    # backtest-results commit behind the report

def sh(*cmd, quiet=False):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if (not quiet or r.returncode) and (r.stdout + r.stderr).strip():
        print((r.stdout + r.stderr).strip())
    if r.returncode:
        raise RuntimeError(f"failed: {' '.join(str(c) for c in cmd)}")
    return r.stdout

def fetch(folder, commit, branch):
    """One commit of the repository (or the tip of a branch) into a folder."""
    if not os.path.exists(folder):
        sh("git", "init", "-q", folder)
        sh("git", "-C", folder, "fetch", "-q", "--depth", "1", REPO, branch if LATEST else commit)
        sh("git", "-C", folder, "checkout", "-q", "FETCH_HEAD")

if not os.path.exists("matchday"):
    sh("git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, "matchday")
fetch("source", SOURCE_COMMIT, "source-data")
fetch("published", RESULTS_COMMIT, "backtest-results")
PIPE = "matchday/analysis/pipeline"
sys.path.insert(0, PIPE)
print("code:", sh("git", "-C", "matchday", "log", "-1", "--format=%h %s", quiet=True).strip())
print("source data:", sh("git", "-C", "source", "log", "-1", "--format=%h %s", quiet=True).strip())
print("forecasts:", sh("git", "-C", "published", "log", "-1", "--format=%h %s", quiet=True).strip())

### One match table per league

Each row keeps the early price (`avg_*`, `b365_*`), the closing price (`avgc_*`) and a pointer to its source file and line, which is how the best early price is looked up later.

In [ ]:
import pandas as pd
os.makedirs("tables", exist_ok=True)
rows = []
for league in LEAGUES:
    sh(sys.executable, f"{PIPE}/build_matches.py", "source", f"tables/{league}.csv", "--league", league, quiet=True)
    m = pd.read_csv(f"tables/{league}.csv", parse_dates=["date"])
    test = m[(m["div"] == "D1") & (m.season >= 2019)]
    rows.append(dict(league=league, test_matches=len(test), with_early_price=int(test.avg_h.notna().sum()),
                     with_closing_price=int(test.avgc_h.notna().sum()), last_match=f"{m.date.max():%Y-%m-%d}"))
pd.DataFrame(rows)

### Grades

For every match and outcome, with the bookmakers' margin removed proportionally:

- **edge** = the ensemble's probability minus the early market's;
- **line move** = the closing probability minus the early one;
- **CLV** of a bet = odds taken × closing probability − 1. Zero is a fair bet if the closing market is right; a blind bet scores about minus the margin.

The slope of line move on edge says how much of the gap the market closed: 0 means it ignored the models, 1 means it came round completely. Intervals are a bootstrap over matches.

In [ ]:
print(sh(sys.executable, f"{PIPE}/evaluate_clv.py", "tables", "source", "published/leagues", "results.json",
         "--xg-results", "published/xg", quiet=True))
res = json.load(open("results.json"))
pub = json.load(open("matchday/analysis/backtest/clv/results.json"))
mine = (res["q1"]["slope"], res["q2"]["clv"]["mean"], res["q2"]["realised_return"]["mean"], res["q2"]["bets"])
theirs = (pub["q1"]["slope"], pub["q2"]["clv"]["mean"], pub["q2"]["realised_return"]["mean"], pub["q2"]["bets"])
same = all(abs(a - b) < 1e-9 for a, b in zip(mine, theirs))
print("Same answers as the published report." if same else
      "Differs from the published report (expected with LATEST = True):", "" if same else (mine, theirs))

### The rule, the mirror rule and betting on everything

The site's rule puts €1 on every outcome the ensemble rates at least 5 points above the market. The mirror rule bets where the ensemble is at least 5 points *below*. "Every bet" is all three outcomes of every match: what no skill at all earns.

In [ ]:
A = res["five_leagues"]
def row(label, r):
    f = lambda k, d=2: f"{r[k]['mean'] * 100:+.{d}f}% ({r[k]['low'] * 100:+.{d}f} to {r[k]['high'] * 100:+.{d}f})"
    return {"bets": label, "number": r["bets"], "average odds": round(r["odds"], 2), "CLV": f("clv"),
            "realised return": f("realised_return", 1), "won": f"{r['won']:.1%}", "closing market said": f"{r['closing']:.1%}"}
pd.DataFrame([row("The site's rule, average early price", A["rule"]["avg"]),
              row("The rule at Bet365's early price", A["rule"]["b365"]),
              row("The rule at the best early price", A["rule"]["max"]),
              row("The mirror rule, average early price", A["mirror"]["avg"]),
              row("Every possible bet, average early price", A["every_bet"]["avg"]),
              row("Bundesliga, published models: the rule", res["bundesliga_published"]["rule"]["avg"])])

In [ ]:
from IPython.display import SVG, display
sh(sys.executable, f"{PIPE}/make_clv_figures.py", "results.json", "figs", quiet=True)
for name in ["line_move", "returns", "returns_by_league"]:
    display(SVG(f"figs/{name}.svg"))

### Check it yourself

The same headline numbers in a dozen lines of pandas, without `evaluate_clv.py`: the early and closing prices, the ensemble's forecasts, and the rule.

In [ ]:
import numpy as np
from evaluate import cols, load
o = pd.concat([load(f"tables/{league}.csv", f"published/leagues/{league}") for league in LEAGUES])
o = o.dropna(subset=cols("bay") + cols("avgc") + cols("avg"))
model, early, closing = (o[cols(k)].to_numpy() for k in ("ens", "avg", "avgc"))
odds = o[["avg_h", "avg_d", "avg_a"]].to_numpy()
won = o.y.to_numpy()[:, None] == np.arange(3)
edge, move = model - early, closing - early
picks = edge >= 0.05 - 1e-9
print(f"{len(o):,} matches")
print(f"slope of line move on edge: {(edge * move).sum() / (edge ** 2).sum():+.4f}")
print(f"the rule: {picks.sum():,} bets, CLV {(odds[picks] * closing[picks] - 1).mean():+.2%}, "
      f"return {(odds[picks] * won[picks] - 1).mean():+.1%}")
print(f"every bet: CLV {(odds * closing - 1).mean():+.2%}, return {(odds * won - 1).mean():+.1%}")